# 📧 Detector de Spam
## Procesamiento de Lenguaje Natural + Clasificación

---

En este ejercicio vamos a construir un **detector de spam** que aprende a distinguir mensajes legítimos de correo basura, igual que lo hace tu casilla de email.

Este es un problema de **Procesamiento de Lenguaje Natural (NLP)**: el modelo aprende a partir de texto, no de números.

### 🧠 ¿Cómo funciona?
Las computadoras no entienden texto directamente. Necesitamos convertir las palabras en números. Para eso usaremos dos técnicas:

| Técnica | Idea |
|---|---|
| **Bag of Words** | Cuenta cuántas veces aparece cada palabra en el mensaje |
| **TF-IDF** | Le da más peso a palabras raras e importantes, y menos a palabras comunes |

### 🎯 Objetivos:
- Entender cómo se trabaja con texto en ML
- Convertir texto en vectores numéricos
- Entrenar y comparar clasificadores
- Analizar qué palabras delatan al spam
- Detectar si un mensaje nuevo es spam

### 🗺️ Estructura:
1. Importar librerías
2. Crear el dataset
3. Explorar los mensajes
4. Preprocesar el texto
5. Entrenar modelos
6. Evaluar y comparar
7. Analizar palabras clave
8. Predecir mensajes nuevos
9. 🏆 Desafíos extra

---
## 📦 Paso 1: Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# NLP
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Modelos
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

# Evaluación
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

np.random.seed(42)
print('✅ Librerías importadas correctamente')

---
## 📂 Paso 2: Crear el dataset

Vamos a usar un conjunto de mensajes en español: algunos legítimos (**ham**) y otros spam.

In [ ]:
# Mensajes spam (correo no deseado)
mensajes_spam = [
    "GANASTE un premio de $10.000! Hacé clic AQUÍ para reclamar tu dinero GRATIS ahora",
    "Oferta exclusiva: comprá 2 y llevate 3 GRATIS. Solo por hoy! No te lo pierdas",
    "Tu cuenta fue suspendida. Verificá tus datos URGENTE haciendo clic en este enlace",
    "Felicitaciones! Sos el ganador del sorteo. Envianos tus datos bancarios para transferirte",
    "Gana dinero desde casa sin esfuerzo. Más de $5000 por semana garantizado",
    "URGENTE: Tu tarjeta de crédito fue bloqueada. Confirmá tu identidad ahora",
    "Préstamo rápido aprobado! Sin garantías ni papeles. Dinero en tu cuenta en 24hs",
    "Bajá 10kg en 2 semanas con esta pastilla milagrosa. Oferta limitada 70% descuento",
    "Invertí $100 y ganás $1000 por día. Sistema probado. Únete gratis hoy",
    "Premio: iPhone 15 GRATIS para los primeros 100 usuarios. Registrate ya",
    "Tu paquete no pudo entregarse. Pagá el arancel de $2 haciendo clic aquí",
    "Conocé solteras/os cerca tuyo. Registrate GRATIS y encontrá el amor hoy",
    "Alerta de seguridad: acceso sospechoso detectado. Cambiá tu contraseña urgente",
    "Cryptocurrency: invertí ahora y multiplicá tu dinero x10 en 30 días garantizado",
    "Conseguí seguidores reales en Instagram: 10.000 seguidores por solo $5",
    "Trabajo desde casa: ganás $500 diarios respondiendo encuestas. Inscripción gratis",
    "IMPORTANTE: Renovar suscripción de Netflix. Tu acceso expira en 24 horas",
    "Medicamentos originales sin receta. Envío discreto. Precios increíbles",
    "Sos el visitante número 1.000.000 de nuestro sitio. Reclamá tu premio ahora",
    "Oferta imperdible: viaje a Cancún para 2 personas por solo $99. Solo hoy!",
    "Tu CVU fue comprometido. Verificá tus movimientos bancarios de inmediato",
    "Ganá la lotería con nuestro sistema infalible. Resultados comprobados 100%",
    "Ampliá tu negocio! Base de datos con 50.000 emails verificados por $10",
    "Alerta: virus detectado en tu teléfono. Descargá nuestro antivirus GRATIS",
    "Hacete millonario invirtiendo en acciones seleccionadas por IA. Empezá hoy",
    "Tu número fue seleccionado para ganar un auto 0km. Confirmá tu participación",
    "Descuento exclusivo 80% en ropa de marca. Solo para suscriptores. Comprá ahora",
    "Transferencia pendiente de $3500 a tu nombre. Verificá tus datos para acreditar",
    "Aprendé a hackear sistemas y ganá miles trabajando desde casa. Curso gratis",
    "No podemos entregar tu pedido. Actualizá tu dirección urgente haciendo clic",
    "Duplicá tus ahorros en bitcoins. Sin riesgo. Retiro inmediato garantizado",
    "GRATIS: Curso de trading que te hará ganar $2000 semanales. Registrate ya",
    "Tu préstamo por $50.000 fue pre-aprobado. Sin consultar Veraz. Pedilo ahora",
    "Oferta especial solo para vos: 3 meses de Spotify Premium completamente gratis",
    "Ganaste un voucher de $5000 en supermercados. Ingresá el código para activarlo",
]

# Mensajes ham (legítimos)
mensajes_ham = [
    "Hola! Quedamos para el martes a las 18hs en el café de siempre?",
    "Recordatorio: reunión de equipo mañana a las 10. Traé el informe del trimestre",
    "Mamá, llegué bien. Mañana te llamo para contarte cómo me fue en el viaje",
    "La factura del mes de marzo está adjunta. Cualquier consulta, avisame",
    "Te mando el documento que me pediste. Revisalo y decime si está bien",
    "El partido empieza a las 21. Lo vemos en lo de Rodrigo o en el bar?",
    "Pasé por la librería y no tenían el libro. Lo pedí por internet, llega el jueves",
    "Buen día! Confirmás asistencia a la capacitación del viernes?",
    "Gracias por tu ayuda con el proyecto. Quedó muy bien el informe final",
    "Necesito que me envíes el presupuesto actualizado antes del mediodía",
    "Feliz cumpleaños! Espero que la estés pasando muy bien con tu familia",
    "El médico confirmó el turno para el miércoles a las 9. No te olvides el carnet",
    "Adjunto el acta de la última reunión. Por favor revisala y confirmá si está ok",
    "Hola profe, no voy a poder asistir a clase mañana por un problema de salud",
    "El informe de ventas del mes pasado muestra un crecimiento del 8% respecto al anterior",
    "¿Podés traer el cargador del notebook? Me lo olvidé en casa hoy",
    "La reunión de padres es el próximo martes a las 19hs en la escuela",
    "Aviso que el local va a estar cerrado el lunes por feriado nacional",
    "Me confirmás si podés venir a la cena del sábado? Somos 8 personas",
    "Revisé el contrato y todo está en orden. Podemos firmar cuando quieras",
    "El pedido número 4521 fue despachado. Llega en 3 a 5 días hábiles",
    "Buen día, les recordamos que el pago del alquiler vence el próximo viernes",
    "Hola! Ya terminé el trabajo práctico. Te lo mando para que le des una mirada",
    "Mañana hay paro de transporte. Avisá si van a trabajar desde casa o no",
    "Te llamo más tarde, ahora estoy en una reunión. Dejame un mensaje si es urgente",
    "El departamento ya está disponible para visitar. ¿Cuándo te viene bien?",
    "Muchas gracias por la atención. El servicio estuvo excelente como siempre",
    "Mañana es el último día para renovar la matrícula. No te olvides de hacer el trámite",
    "Adjunto el cronograma de actividades para el próximo mes. Cualquier cambio, avisame",
    "Hola, ¿siguen vendiendo el modelo azul de la campera? Quería comprarla para el fin de semana",
    "La próxima clase de yoga es el jueves a las 8. Traé tu mat y ropa cómoda",
    "Confirmamos tu reserva para el 15 de julio. Podés hacer el check-in desde las 14hs",
    "El libro que me recomendaste está buenísimo. Ya voy por la mitad y no puedo parar",
    "Necesito el número de CUIT de la empresa para completar la declaración jurada",
    "Hola! El grupo de estudio se junta el domingo a las 16 en la biblioteca",
]

# Crear DataFrame
mensajes = mensajes_spam + mensajes_ham
etiquetas = ['spam'] * len(mensajes_spam) + ['ham'] * len(mensajes_ham)

df = pd.DataFrame({'mensaje': mensajes, 'etiqueta': etiquetas})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # mezclar
df['etiqueta_num'] = (df['etiqueta'] == 'spam').astype(int)

print(f'📊 Dataset creado: {len(df)} mensajes')
print(df['etiqueta'].value_counts())
print('\nEjemplo de mensajes:')
print(df[['etiqueta', 'mensaje']].head(6).to_string(index=False))

---
## 🔍 Paso 3: Explorar los mensajes

In [ ]:
# Longitud de los mensajes
df['longitud']  = df['mensaje'].apply(len)
df['palabras']  = df['mensaje'].apply(lambda x: len(x.split()))
df['mayusculas'] = df['mensaje'].apply(lambda x: sum(1 for c in x if c.isupper()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Características de los mensajes: Spam vs Ham', fontsize=13, fontweight='bold')

colores = {'spam': '#E74C3C', 'ham': '#2ECC71'}

for ax, (col, titulo) in zip(axes, [
    ('longitud',   'Longitud (caracteres)'),
    ('palabras',   'Cantidad de palabras'),
    ('mayusculas', 'Letras mayúsculas'),
]):
    for etiq, color in colores.items():
        subset = df[df['etiqueta'] == etiq][col]
        ax.hist(subset, bins=15, alpha=0.6, color=color, label=etiq, edgecolor='white')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Cantidad')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Estadísticas por tipo de mensaje:')
print(df.groupby('etiqueta')[['longitud','palabras','mayusculas']].mean().round(1))

In [ ]:
# Palabras más frecuentes en spam vs ham
def palabras_frecuentes(textos, n=15):
    """Devuelve las N palabras más frecuentes (ignorando palabras cortas)."""
    todas = ' '.join(textos).lower()
    palabras = re.findall(r'\b[a-záéíóúñü]{4,}\b', todas)
    return Counter(palabras).most_common(n)

freq_spam = palabras_frecuentes(df[df['etiqueta']=='spam']['mensaje'])
freq_ham  = palabras_frecuentes(df[df['etiqueta']=='ham']['mensaje'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Palabras más frecuentes', fontsize=13, fontweight='bold')

for ax, freq, titulo, color in [
    (axes[0], freq_spam, '📩 SPAM', '#E74C3C'),
    (axes[1], freq_ham,  '✉️  HAM',  '#2ECC71'),
]:
    palabras_v = [p for p, _ in freq]
    conteos    = [c for _, c in freq]
    ax.barh(palabras_v[::-1], conteos[::-1], color=color, edgecolor='white')
    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_xlabel('Frecuencia')
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## ⚙️ Paso 4: Preprocesar el texto

Convertimos el texto en vectores numéricos con dos técnicas:

- **Bag of Words (BoW)**: cada mensaje se representa por cuántas veces aparece cada palabra
- **TF-IDF**: pondera las palabras por su importancia relativa en el corpus

In [ ]:
X = df['mensaje']
y = df['etiqueta_num']

# Dividir antes de vectorizar (evitar data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Bag of Words
bow = CountVectorizer(lowercase=True, strip_accents='unicode', min_df=1)
X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer(lowercase=True, strip_accents='unicode',
                        ngram_range=(1, 2), min_df=1)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('✅ Texto convertido a vectores numéricos')
print(f'   BoW   → dimensión del vocabulario: {X_train_bow.shape[1]} palabras')
print(f'   TF-IDF → dimensión del vocabulario: {X_train_tfidf.shape[1]} términos (incl. bigramas)')
print(f'\n   Entrenamiento: {X_train_bow.shape[0]} mensajes')
print(f'   Prueba:        {X_test_bow.shape[0]} mensajes')

In [ ]:
# Visualizar cómo se ve un mensaje como vector
msg_ejemplo = X_train.iloc[0]
vec_ejemplo = bow.transform([msg_ejemplo]).toarray()[0]
palabras_presentes = [(bow.get_feature_names_out()[i], vec_ejemplo[i])
                      for i in np.where(vec_ejemplo > 0)[0]]

print(f'Mensaje: "{msg_ejemplo}"')
print(f'\nRepresentación Bag of Words (solo palabras presentes):')
for p, v in palabras_presentes:
    print(f'   "{p}" → {int(v)}')
print(f'\nEl vector tiene {len(bow.get_feature_names_out())} posiciones en total,'
      f' pero solo {len(palabras_presentes)} son distintas de 0.')

---
## 🤖 Paso 5: Entrenar modelos

Vamos a entrenar **4 clasificadores** con representación **TF-IDF** (generalmente la mejor para texto).

In [ ]:
modelos = {
    'Naive Bayes':          MultinomialNB(),
    'Regresión Logística':  LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM Lineal':           LinearSVC(random_state=42, max_iter=2000),
}

resultados = {}

print(f'{"Modelo":<25}  {"Accuracy":>10}  {"Precisión":>10}  {"Recall":>8}  {"F1":>8}')
print('-' * 68)

for nombre, modelo in modelos.items():
    modelo.fit(X_train_tfidf, y_train)
    y_pred = modelo.predict(X_test_tfidf)

    from sklearn.metrics import precision_score, recall_score, f1_score
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    resultados[nombre] = {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'y_pred': y_pred}
    print(f'{nombre:<25}  {acc*100:>9.1f}%  {prec*100:>9.1f}%  {rec*100:>7.1f}%  {f1*100:>7.1f}%')

mejor = max(resultados, key=lambda k: resultados[k]['f1'])
print(f'\n🏆 Mejor modelo (mayor F1): {mejor}')

---
## 📊 Paso 6: Evaluar y comparar

In [ ]:
# Comparación visual de métricas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparación de modelos', fontsize=13, fontweight='bold')

nombres = list(resultados.keys())
cols_mod = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6']

# Accuracy
accs = [resultados[n]['acc'] * 100 for n in nombres]
bars = axes[0].bar(nombres, accs, color=cols_mod, edgecolor='white')
axes[0].set_title('Accuracy (%)', fontweight='bold')
axes[0].set_ylim([0, 115])
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{v:.1f}%', ha='center', fontweight='bold')

# F1-Score
f1s = [resultados[n]['f1'] * 100 for n in nombres]
bars = axes[1].bar(nombres, f1s, color=cols_mod, edgecolor='white')
axes[1].set_title('F1-Score (%)', fontweight='bold')
axes[1].set_ylim([0, 115])
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 En detección de spam, el F1-Score es más importante que el Accuracy.')
print('   Combina Precisión (no marcar ham como spam) y Recall (no dejar pasar spam).')

In [ ]:
# Matriz de confusión del mejor modelo
y_pred_mejor = resultados[mejor]['y_pred']
cm = confusion_matrix(y_test, y_pred_mejor)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham (predicho)', 'Spam (predicho)'],
            yticklabels=['Ham (real)', 'Spam (real)'], ax=ax)
ax.set_title(f'Matriz de Confusión — {mejor}', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\n   Verdaderos Negativos (ham correcto):   {tn}')
print(f'   Falsos Positivos (ham marcado spam):   {fp}  ← ERROR grave (perdemos un email legítimo)')
print(f'   Falsos Negativos (spam no detectado):  {fn}  ← error tolerable')
print(f'   Verdaderos Positivos (spam correcto):  {tp}')

In [ ]:
# Reporte completo
print(f'Reporte de clasificación — {mejor}:\n')
print(classification_report(y_test, y_pred_mejor, target_names=['Ham', 'Spam']))

---
## 🔑 Paso 7: Analizar palabras clave del spam

¿Qué palabras usa el modelo para detectar spam?

In [ ]:
# Palabras más indicativas de spam y ham (usando Regresión Logística)
lr = modelos['Regresión Logística']
feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]

# Top palabras que indican SPAM (coeficiente positivo alto)
top_spam_idx = np.argsort(coefs)[::-1][:20]
top_ham_idx  = np.argsort(coefs)[:20]

top_spam_words = [(feature_names[i], coefs[i]) for i in top_spam_idx]
top_ham_words  = [(feature_names[i], coefs[i]) for i in top_ham_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Palabras más indicativas para el modelo', fontsize=13, fontweight='bold')

for ax, words, titulo, color in [
    (axes[0], top_spam_words, '🚫 Indican SPAM',  '#E74C3C'),
    (axes[1], top_ham_words,  '✅ Indican HAM',   '#2ECC71'),
]:
    pals   = [w for w, _ in words]
    scores = [abs(c) for _, c in words]
    ax.barh(pals[::-1], scores[::-1], color=color, edgecolor='white')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Importancia del coeficiente')
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🔮 Paso 8: Predecir mensajes nuevos

In [ ]:
def detectar_spam(mensaje, modelo_nombre=None):
    """
    Detecta si un mensaje es spam o ham.

    Parámetros:
        mensaje       : texto del mensaje a analizar
        modelo_nombre : nombre del modelo a usar (por defecto: el mejor)
    """
    if modelo_nombre is None:
        modelo_nombre = mejor

    modelo_sel = modelos[modelo_nombre]
    vec = tfidf.transform([mensaje])
    pred = modelo_sel.predict(vec)[0]

    etiqueta = 'SPAM 🚫' if pred == 1 else 'HAM ✅'
    emoji    = '🚨' if pred == 1 else '✉️ '

    print(f'{emoji} Mensaje: "{mensaje[:80]}{'...' if len(mensaje)>80 else ''}"')
    print(f'   Clasificación: {etiqueta}')

    # Probabilidad (si el modelo lo soporta)
    if hasattr(modelo_sel, 'predict_proba'):
        prob = modelo_sel.predict_proba(vec)[0]
        print(f'   Confianza: Ham={prob[0]*100:.1f}%  |  Spam={prob[1]*100:.1f}%')
    print()


print('=== PREDICCIONES DE EJEMPLO ===\n')

detectar_spam("Ganaste un iPhone GRATIS! Hacé clic para reclamarlo ahora mismo")
detectar_spam("Hola! ¿Podemos juntarnos mañana a las 18 para repasar el trabajo?")
detectar_spam("URGENTE: Tu cuenta bancaria fue suspendida. Verificá tus datos ya")
detectar_spam("El informe del trimestre está listo. Te lo mando por mail hoy")
detectar_spam("Invertí solo $50 y ganá $5000 por semana. Sistema 100% garantizado")

In [ ]:
# 🔧 ¡Probá con tu propio mensaje!

MI_MENSAJE = "Recordá que mañana es el examen. Traé el documento y llegá 10 minutos antes"  # ← modificá

detectar_spam(MI_MENSAJE)

---
## 🏆 Desafíos Extra

---
### 🥉 Desafío 1 (Fácil): Bag of Words vs TF-IDF

¿Cuál representación es mejor para detectar spam?

In [ ]:
# DESAFÍO 1: Comparar BoW vs TF-IDF con Naive Bayes
from sklearn.metrics import f1_score

nb_bow   = MultinomialNB()
nb_tfidf = MultinomialNB()

nb_bow.fit(X_train_bow, y_train)
nb_tfidf.fit(X_train_tfidf, y_train)

f1_bow   = f1_score(y_test, nb_bow.predict(X_test_bow))
f1_tfidf = f1_score(y_test, nb_tfidf.predict(X_test_tfidf))

print('Naive Bayes con distintas representaciones:')
print(f'   Bag of Words → F1: {f1_bow*100:.1f}%')
print(f'   TF-IDF       → F1: {f1_tfidf*100:.1f}%')
ganador = 'TF-IDF' if f1_tfidf >= f1_bow else 'Bag of Words'
print(f'\n🏆 Mejor representación: {ganador}')

---
### 🥈 Desafío 2 (Medio): Agregar tus propios mensajes al dataset

Ampliá el dataset con mensajes propios, reentrená y observá si mejora.

In [ ]:
# DESAFÍO 2: Agregar mensajes propios

# ← Agregá tus propios mensajes acá
nuevos_spam = [
    "Oferta única: seguro de vida sin examen médico. Cobertura inmediata por $1 al día",
    "Tu solicitud de préstamo fue APROBADA. Retirá hasta $100.000 en efectivo hoy",
]

nuevos_ham = [
    "Buen día, te recuerdo que el informe vence el viernes a las 17hs",
    "Hola! Ya cargué los cambios al repositorio. Revisalo cuando puedas",
]

# Agregar al dataset
df_extra = pd.DataFrame({
    'mensaje':     nuevos_spam + nuevos_ham,
    'etiqueta':    ['spam'] * len(nuevos_spam) + ['ham'] * len(nuevos_ham),
    'etiqueta_num': [1] * len(nuevos_spam) + [0] * len(nuevos_ham),
})

df_ampliado = pd.concat([df[['mensaje','etiqueta','etiqueta_num']], df_extra], ignore_index=True)
df_ampliado = df_ampliado.sample(frac=1, random_state=42).reset_index(drop=True)

# Reentrenar
X2 = df_ampliado['mensaje']
y2 = df_ampliado['etiqueta_num']
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

tfidf2 = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1,2))
X2_tr_v = tfidf2.fit_transform(X2_tr)
X2_te_v = tfidf2.transform(X2_te)

lr2 = LogisticRegression(max_iter=1000, random_state=42)
lr2.fit(X2_tr_v, y2_tr)
f1_nuevo = f1_score(y2_te, lr2.predict(X2_te_v))
f1_orig  = resultados['Regresión Logística']['f1']

print(f'Dataset original  → F1: {f1_orig*100:.1f}%  ({len(df)} mensajes)')
print(f'Dataset ampliado  → F1: {f1_nuevo*100:.1f}%  ({len(df_ampliado)} mensajes)')
print(f'Cambio: {(f1_nuevo - f1_orig)*100:+.1f}%')

---
### 🥇 Desafío 3 (Difícil): Umbral de decisión

Por defecto, el modelo clasifica como spam si la probabilidad es > 50%. ¿Qué pasa si cambiamos ese umbral?

In [ ]:
# DESAFÍO 3: Análisis del umbral de decisión
from sklearn.metrics import precision_score, recall_score

nb = modelos['Naive Bayes']
probs_spam = nb.predict_proba(X_test_tfidf)[:, 1]

umbrales   = np.arange(0.1, 1.0, 0.05)
precisiones, recalls, f1s = [], [], []

for umbral in umbrales:
    y_pred_u = (probs_spam >= umbral).astype(int)
    precisiones.append(precision_score(y_test, y_pred_u, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_u, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_u, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(umbrales, precisiones, label='Precisión',  color='#3498DB', linewidth=2)
ax.plot(umbrales, recalls,     label='Recall',     color='#E74C3C', linewidth=2)
ax.plot(umbrales, f1s,         label='F1-Score',   color='#2ECC71', linewidth=2, linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='Umbral default (0.5)')
mejor_u = umbrales[np.argmax(f1s)]
ax.axvline(mejor_u, color='orange', linestyle='--', label=f'Mejor umbral ({mejor_u:.2f})')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Valor de la métrica')
ax.set_title('Precisión, Recall y F1 según el umbral (Naive Bayes)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'🏆 Mejor umbral para F1: {mejor_u:.2f}  →  F1: {max(f1s)*100:.1f}%')
print('\n💡 Subir el umbral reduce los falsos positivos (ham clasificado como spam)')
print('   pero deja pasar más spam. Bajarlo hace lo contrario.')

---
## 📝 Resumen y Conclusiones

### Lo que aprendiste:

| Concepto | Descripción |
|---|---|
| **NLP** | Procesamiento de Lenguaje Natural: trabajar con texto en ML |
| **Bag of Words** | Representar texto contando apariciones de palabras |
| **TF-IDF** | Dar más peso a palabras raras e importantes |
| **Naive Bayes** | Clasificador probabilístico muy usado en texto |
| **Precisión** | De los que clasifiqué como spam, ¿cuántos realmente lo son? |
| **Recall** | De todos los spams, ¿cuántos detecté? |
| **F1-Score** | Balance entre precisión y recall |
| **Umbral de decisión** | Ajustar el punto de corte según la tolerancia al error |

### Para seguir aprendiendo:
- 📚 Probá con el dataset real SMS Spam Collection de UCI
- 🔍 Investigá **NLTK** y **spaCy** para preprocesamiento de texto más avanzado
- 🎓 Explorá modelos de lenguaje modernos: **BERT**, **transformers**